# Setup Môi Trường & BoT-SORT

In [4]:
# === CELL 1: Setup Môi Trường ===
import subprocess, os, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "filterpy", "lap", "cython-bbox", "yacs", "loguru", "easydict", "faiss-cpu", "ultralytics"])

if not os.path.exists('/kaggle/working/BoT-SORT'):
    subprocess.run(["git", "clone", "https://github.com/niraharon/bot-sort.git", "/kaggle/working/BoT-SORT"])

sys.path.insert(0, '/kaggle/working/BoT-SORT')
# Tải mô hình ReID
if not os.path.exists('/kaggle/working/BoT-SORT/pretrained/mot17_sbs_S50.pth'):
    os.makedirs('/kaggle/working/BoT-SORT/pretrained', exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    subprocess.run(["gdown", "1QZFWpoa80rqo7O-HXmlss8J8CnS7IUsN", "-O", "/kaggle/working/BoT-SORT/pretrained/mot17_sbs_S50.pth"])

# Tải mô hình ReID cho Xe cộ (VeRi)
if not os.path.exists('/kaggle/working/BoT-SORT/pretrained/veri_sbs_R50-ibn.pth'):
    subprocess.run(["wget", "-nc", "https://github.com/JDAI-CV/fast-reid/releases/download/v0.1.1/veri_sbs_R50-ibn.pth", "-O", "/kaggle/working/BoT-SORT/pretrained/veri_sbs_R50-ibn.pth"])


### KEEP LABELS

In [5]:
KEEP_LABELS = [
    "car_(automobile)",
    "motorcycle",
    "truck",
    "bus_(vehicle)",
    "minivan",
    "motor_scooter",
    "cab_(taxi)",
    "pickup_truck",
    "garbage_truck",
    "camper_(vehicle)",
    "bicycle",
    "unicycle",
    "tricycle",
    "skateboard",
    "baby_buggy",
    "cart",
    "handcart",
    "horse_carriage",
    "horse_buggy",
    "golfcart",
    "wheelchair",
    "wagon",
    "boat",
    "houseboat",
    "airplane",
    "hot-air_balloon",
    "horse",
    "elephant",
    "cow",
    "hippopotamus",
    "mammoth",
    "lion",
    "tiger",
    "traffic_light",
    "street_sign",
    "signboard",
    "stop_sign",
    "billboard",
    "banner",
    "poster",
    "pole",
    "telephone_pole",
    "streetlight",
    "lamppost",
    "flagpole",
    "manhole",
    "cone",
    "fireplug",
    "parking_meter",
    "crossbar",
    "sawhorse",
    "water_tower",
    "reflector",
    "person",
    "statue_(sculpture)",
    "sculpture",
    "bench",
    "chair",
    "stool",
    "trash_can",
    "postbox_(public)",
    "vending_machine",
    "telephone_booth",
    "newsstand",
    "pew_(church_bench)",
    "box",
    "crate",
    "barrel",
    "suitcase",
    "cabinet",
    "canister",
    "pipe",
    "hose",
    "garden_hose",
    "ladder",
    "stepladder",
    "easel",
    "flowerpot",
    "window_box_(for_plants)",
    "tarp",
    "log",
    "bamboo",
    "fan",
    "spotlight",
    "lamp",
    "awning",
    "flag",
    "clock_tower",
    "blackboard",
    "television_set",
    "painting",
    "flower_arrangement",
    "Christmas_tree",
    "gravestone",
    "dining_table",
    "table",
    "refrigerator",
    "mirror",
    "umbrella",
    "dog",
    "cat",
    "ball",
    "beachball",
    "tambourine",
    "shield",
    "shopping_cart",
    "train_(railroad_vehicle)",
]
print("Số lượng KEEP_LABELS:" ,len(KEEP_LABELS))


Số lượng KEEP_LABELS: 107


## Fix BoTSORT

In [6]:
# === CELL 1.5: Vá mã nguồn lõi BoT-SORT (Fix matching.py) ===
import os
import numpy as np

matching_path = '/kaggle/working/BoT-SORT/tracker/matching.py'
if os.path.exists(matching_path):
    with open(matching_path, 'r', encoding='utf-8') as f:
        matching_code = f.read()

    target_matching = '''def embedding_distance(tracks, detections, metric='cosine'):
    """
    :param tracks: list[STrack]
    :param detections: list[BaseTrack]
    :param metric:
    :return: cost_matrix np.ndarray
    """

    cost_matrix = np.zeros((len(tracks), len(detections)), dtype=np.float)
    if cost_matrix.size == 0:
        return cost_matrix
    det_features = np.asarray([track.curr_feat for track in detections], dtype=np.float)
    track_features = np.asarray([track.smooth_feat for track in tracks], dtype=np.float)

    cost_matrix = np.maximum(0.0, cdist(track_features, det_features, metric))  # / 2.0  # Nomalized features
    return cost_matrix'''

    replacement_matching = '''def embedding_distance(tracks, detections, metric='cosine'):
    """
    :param tracks: list[STrack]
    :param detections: list[BaseTrack]
    :param metric:
    :return: cost_matrix np.ndarray
    """

    cost_matrix = np.full((len(tracks), len(detections)), 2.0, dtype=np.float)
    if cost_matrix.size == 0:
        return cost_matrix

    valid_track_idx = [i for i, t in enumerate(tracks) if getattr(t, 'smooth_feat', None) is not None]
    valid_det_idx = [i for i, d in enumerate(detections) if getattr(d, 'curr_feat', None) is not None]

    if len(valid_track_idx) == 0 or len(valid_det_idx) == 0:
        return cost_matrix

    valid_track_features = np.asarray([tracks[i].smooth_feat for i in valid_track_idx], dtype=np.float)
    valid_det_features = np.asarray([detections[i].curr_feat for i in valid_det_idx], dtype=np.float)

    sub_cost_matrix = np.maximum(0.0, cdist(valid_track_features, valid_det_features, metric))

    for i, trk_idx in enumerate(valid_track_idx):
        for j, det_idx in enumerate(valid_det_idx):
            cost_matrix[trk_idx, det_idx] = sub_cost_matrix[i, j]

    return cost_matrix'''

    if target_matching in matching_code or target_matching.replace('\n', '\r\n') in matching_code:
        matching_code = matching_code.replace(target_matching, replacement_matching)
        matching_code = matching_code.replace(target_matching.replace('\n', '\r\n'), replacement_matching)
        with open(matching_path, 'w', encoding='utf-8') as f:
            f.write(matching_code)
        print("Đã vá xong matching.py")
    else:
        print("matching.py đã được vá hoặc không tìm thấy chuỗi cần thay thế.")

Đã vá xong matching.py


### Sửa lỗi Python 3.10+ và NumPy, PyTorch Mismatch

In [7]:
# === CELL 2: Monkey-Patch Tương Thích (KHÔNG ĐỔI) ===
import collections
import collections.abc
import numpy as np
import sys
import types
import torch

collections.Mapping = collections.abc.Mapping
collections.MutableMapping = collections.abc.MutableMapping

if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'bool'):
    np.bool = bool
if not hasattr(np, 'asfarray'):
    np.asfarray = lambda x: np.asarray(x, dtype=float)

if 'torch._six' not in sys.modules:
    dummy_six = types.ModuleType('torch._six')
    dummy_six.string_classes = str
    sys.modules['torch._six'] = dummy_six
    if not hasattr(torch, '_six'):
        torch._six = dummy_six

print('Đã kích hoạt monkey-patch tương thích môi trường Python 3.10+ thành công!')

Đã kích hoạt monkey-patch tương thích môi trường Python 3.10+ thành công!


### Sửa lỗi Import FastReID & Class-Aware Gating

In [8]:
# === CELL 3: Fast-ReID Path ===
import sys
import os
BOT_SORT_PATH = '/kaggle/working/BoT-SORT'
if os.path.join(BOT_SORT_PATH, 'fast_reid') not in sys.path:
    sys.path.insert(0, os.path.join(BOT_SORT_PATH, 'fast_reid'))
print('Đã thêm fast_reid vào sys.path!')

Đã thêm fast_reid vào sys.path!


## Import và Các Hàm Bổ Trợ

In [9]:
# === CELL 4: Import + Hàm bổ trợ + Class Gating Monkey-Patch ===
import json
import math
import numpy as np
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict

from tracker.mc_bot_sort import BoTSORT, STrack, joint_stracks, sub_stracks, remove_duplicate_stracks
from tracker.basetrack import BaseTrack, TrackState
from tracker import matching


def compute_focal_length(image_width, fov_deg=90):
    return (image_width / 2) / math.tan(math.radians(fov_deg / 2))

def compute_azimuth_from_bbox(bbox, image_width, f):
    x1, y1, x2, y2 = bbox
    cx = (x1 + x2) / 2.0
    dx = cx - image_width / 2
    return math.degrees(math.atan2(dx, f))

def azimuth_to_clock_10_to_2(azimuth_deg):
    if azimuth_deg <= -30: return "10 o'clock"
    elif azimuth_deg <= -10: return "11 o'clock"
    elif azimuth_deg <= 10: return "12 o'clock"
    elif azimuth_deg <= 30: return "1 o'clock"
    else: return "2 o'clock"

def calculate_iou(box1, box2):
    xx1 = max(box1[0], box2[0])
    yy1 = max(box1[1], box2[1])
    xx2 = min(box1[2], box2[2])
    yy2 = min(box1[3], box2[3])
    w = max(0, xx2 - xx1)
    h = max(0, yy2 - yy1)
    inter = w * h
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def get_motion_info_v2(track, img_w, img_h):
    if track.tracklet_len < 1:
        return {'speed_percent': 0.0, 'angle': 0.0}
    mean = track.mean
    vx_px, vy_px = mean[4], mean[5]
    diagonal = np.sqrt(img_w**2 + img_h**2)
    raw_speed = np.sqrt(vx_px**2 + vy_px**2)
    speed_percent = round((raw_speed / diagonal) * 100, 2)
    angle = np.degrees(np.arctan2(vy_px, vx_px))
    return {'speed_percent': speed_percent, 'angle': round(angle, 2)}

def draw_annotations(img, track_id, label, bbox_px, clock, speed_percent, angle_deg, distance_px, is_new=False):
    x1, y1, x2, y2 = map(int, bbox_px)
    if track_id > 0:
        color = (0, 255, 255) if is_new else (0, 255, 0)
    else:
        color = (128, 128, 128)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
    label_text = f'ID:{track_id} {label}' if track_id > 0 else f'{label}'
    info_text = f'{clock} | {speed_percent}% | {distance_px}'
    cv2.putText(img, label_text, (x1, y1 - 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    cv2.putText(img, info_text, (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    if track_id > 0 and speed_percent > 0.1:
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        rad = math.radians(angle_deg)
        arrow_length = 40
        dx, dy = int(math.cos(rad) * arrow_length), int(math.sin(rad) * arrow_length)
        cv2.arrowedLine(img, (cx, cy), (cx + dx, cy + dy), (0, 0, 255), 3, tipLength=0.35)
    return img

# Class-Aware Gating Monkey-Patch
def apply_class_gating(cost_matrix, tracks, detections, person_classes, vehicle_classes, penalty=2.0):
    if cost_matrix.size == 0:
        return cost_matrix
    def get_super_class(cls_id):
        if cls_id in person_classes: return 0
        elif cls_id in vehicle_classes: return 1
        else: return 2
    for i, track in enumerate(tracks):
        for j, det in enumerate(detections):
            sc_track = get_super_class(track.cls)
            sc_det = get_super_class(det.cls)
            if sc_track != sc_det:
                cost_matrix[i, j] += penalty
            else:
                if sc_track == 2 and track.cls != det.cls:
                    cost_matrix[i, j] += penalty
    return cost_matrix

def _update_with_class_gating(self, output_results, img, shared_warp=None):
    self.frame_id += 1
    activated_starcks = []
    refind_stracks = []
    lost_stracks = []
    removed_stracks = []

    if len(output_results):
        bboxes = output_results[:, :4]
        scores = output_results[:, 4]
        classes = output_results[:, 5]
        features = output_results[:, 6:]

        lowest_inds = scores > self.track_low_thresh
        bboxes = bboxes[lowest_inds]
        scores = scores[lowest_inds]
        classes = classes[lowest_inds]
        features = features[lowest_inds]

        remain_inds = scores > self.args.track_high_thresh
        dets = bboxes[remain_inds]
        scores_keep = scores[remain_inds]
        classes_keep = classes[remain_inds]
        features_keep = features[remain_inds]
        
        actual_indices = np.where(lowest_inds)[0][remain_inds]
    else:
        bboxes = []
        scores = []
        classes = []
        dets = []
        scores_keep = []
        classes_keep = []
        actual_indices = []

    # Extract embeddings (Standard ReID)
    if self.args.with_reid and getattr(self, 'encoder', None) is not None:
        features_keep = self.encoder.inference(img, dets)
    else:
        features_keep = [None] * len(dets)

    if len(dets) > 0:
        detections = []
        for idx, (tlbr, s, c) in enumerate(zip(dets, scores_keep, classes_keep)):
            f = features_keep[idx] if self.args.with_reid else None
            track_obj = STrack(STrack.tlbr_to_tlwh(tlbr), s, c, f)
            track_obj.det_idx = int(actual_indices[idx])
            detections.append(track_obj)
    else:
        detections = []

    unconfirmed = []
    tracked_stracks = []
    for track in self.tracked_stracks:
        if not track.is_activated:
            unconfirmed.append(track)
        else:
            tracked_stracks.append(track)

    strack_pool = joint_stracks(tracked_stracks, self.lost_stracks)
    STrack.multi_predict(strack_pool)

    # GMC Tùy chỉnh (Sử dụng shared_warp nếu có)
    if shared_warp is not None:
        warp = shared_warp
    else:
        warp = self.gmc.apply(img, dets)
        
    STrack.multi_gmc(strack_pool, warp)
    STrack.multi_gmc(unconfirmed, warp)

    ious_dists = matching.iou_distance(strack_pool, detections)
    ious_dists_mask = (ious_dists > self.proximity_thresh)

    if not getattr(self.args, 'mot20', False):
        ious_dists = matching.fuse_score(ious_dists, detections)

    if self.args.with_reid:
        emb_dists = matching.embedding_distance(strack_pool, detections) / 2.0
        emb_dists[emb_dists > self.appearance_thresh] = 1.0
        emb_dists[ious_dists_mask] = 1.0
        dists = np.minimum(ious_dists, emb_dists)
    else:
        dists = ious_dists

    # Class gating 1
    dists = apply_class_gating(dists, strack_pool, detections, getattr(self.args, 'person_class_ids', []), getattr(self.args, 'vehicle_class_ids', []))

    matches, u_track, u_detection = matching.linear_assignment(dists, thresh=self.args.match_thresh)

    for itracked, idet in matches:
        track = strack_pool[itracked]
        det = detections[idet]
        if track.state == TrackState.Tracked:
            track.update(det, self.frame_id)
            track.det_idx = det.det_idx
            activated_starcks.append(track)
        else:
            track.re_activate(det, self.frame_id, new_id=False)
            track.det_idx = det.det_idx
            refind_stracks.append(track)

    if len(scores):
        inds_high = scores < self.args.track_high_thresh
        inds_low = scores > self.args.track_low_thresh
        inds_second = np.logical_and(inds_low, inds_high)
        dets_second = bboxes[inds_second]
        scores_second = scores[inds_second]
        classes_second = classes[inds_second]
        actual_indices_second = np.where(lowest_inds)[0][inds_second]
    else:
        dets_second = []
        scores_second = []
        classes_second = []
        actual_indices_second = []

    if len(dets_second) > 0:
        detections_second = []
        for idx, (tlbr, s, c) in enumerate(zip(dets_second, scores_second, classes_second)):
            track_obj = STrack(STrack.tlbr_to_tlwh(tlbr), s, c)
            track_obj.det_idx = int(actual_indices_second[idx])
            detections_second.append(track_obj)
    else:
        detections_second = []

    r_tracked_stracks = [strack_pool[i] for i in u_track if strack_pool[i].state == TrackState.Tracked]
    dists = matching.iou_distance(r_tracked_stracks, detections_second)

    # Class gating 2
    dists = apply_class_gating(dists, r_tracked_stracks, detections_second, getattr(self.args, 'person_class_ids', []), getattr(self.args, 'vehicle_class_ids', []))

    matches, u_track, u_detection_second = matching.linear_assignment(dists, thresh=0.7)
    for itracked, idet in matches:
        track = r_tracked_stracks[itracked]
        det = detections_second[idet]
        if track.state == TrackState.Tracked:
            track.update(det, self.frame_id)
            track.det_idx = det.det_idx
            activated_starcks.append(track)
        else:
            track.re_activate(det, self.frame_id, new_id=False)
            track.det_idx = det.det_idx
            refind_stracks.append(track)

    for it in u_track:
        track = r_tracked_stracks[it]
        if not track.state == TrackState.Lost:
            track.mark_lost()
            track.det_idx = -1
            lost_stracks.append(track)

    detections = [detections[i] for i in u_detection]
    dists = matching.iou_distance(unconfirmed, detections)
    if not getattr(self.args, 'mot20', False):
        dists = matching.fuse_score(dists, detections)

    # Class gating 3
    dists = apply_class_gating(dists, unconfirmed, detections, getattr(self.args, 'person_class_ids', []), getattr(self.args, 'vehicle_class_ids', []))

    matches, u_unconfirmed, u_detection = matching.linear_assignment(dists, thresh=0.7)
    for itracked, idet in matches:
        unconfirmed[itracked].update(detections[idet], self.frame_id)
        unconfirmed[itracked].det_idx = detections[idet].det_idx
        activated_starcks.append(unconfirmed[itracked])
    for it in u_unconfirmed:
        track = unconfirmed[it]
        track.mark_removed()
        track.det_idx = -1
        removed_stracks.append(track)

    for inew in u_detection:
        track = detections[inew]
        if track.score < self.new_track_thresh:
            continue
        track.activate(self.kalman_filter, self.frame_id)
        activated_starcks.append(track)

    for track in self.lost_stracks:
        if self.frame_id - track.end_frame > self.max_time_lost:
            track.mark_removed()
            track.det_idx = -1
            removed_stracks.append(track)

    self.tracked_stracks = [t for t in self.tracked_stracks if t.state == TrackState.Tracked]
    self.tracked_stracks = joint_stracks(self.tracked_stracks, activated_starcks)
    self.tracked_stracks = joint_stracks(self.tracked_stracks, refind_stracks)
    self.lost_stracks = sub_stracks(self.lost_stracks, self.tracked_stracks)
    self.lost_stracks.extend(lost_stracks)
    self.lost_stracks = sub_stracks(self.lost_stracks, self.removed_stracks)
    self.removed_stracks.extend(removed_stracks)
    self.tracked_stracks, self.lost_stracks = remove_duplicate_stracks(self.tracked_stracks, self.lost_stracks)

    output_stracks = [track for track in self.tracked_stracks]
    return output_stracks

BoTSORT.update = _update_with_class_gating
print('Đã monkey-patch BoTSORT.update() với shared_warp thành công!')


Đã monkey-patch BoTSORT.update() với shared_warp thành công!


# Cấu hình BoT-SORT

In [10]:
# === CELL 5: BoTSORTArgs ===
class BoTSORTArgs:
    def __init__(self):
        # self.track_high_thresh = 0.3
        # self.track_low_thresh = 0.1
        self.track_high_thresh = 0.45
        self.track_low_thresh = 0.1
        self.new_track_thresh = 0.5
        self.match_thresh = 0.7
        self.track_buffer = 30          # Giữ track tối đa 3 frames (1 giây ở 3FPS)
        self.mot20 = False
        self.cmc_method = "sparseOptFlow"
        self.with_reid = True
        import torch
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.person_reid_config = r"/kaggle/working/BoT-SORT/fast_reid/configs/MOT17/sbs_S50.yml"
        self.person_reid_weights = r"/kaggle/working/BoT-SORT/pretrained/mot17_sbs_S50.pth"
        self.vehicle_reid_config = r"/kaggle/working/BoT-SORT/fast_reid/configs/VeRi/sbs_R50-ibn.yml"
        self.vehicle_reid_weights = r"/kaggle/working/BoT-SORT/pretrained/veri_sbs_R50-ibn.pth"
        self.name = "bot_sort"
        self.ablation = False
        self.proximity_thresh = 0.98
        self.appearance_thresh = 0.4

# Định nghĩa Pipeline Xử lý Tracking

In [11]:
# === CELL 6: Pipeline Tracking ===

class DualReIDTrackerWrapper:
    def __init__(self, args_global, person_classes, vehicle_classes):
        import copy
        from tracker.mc_bot_sort import BoTSORT
        from tracker.gmc import GMC
        
        self.person_classes = person_classes
        self.vehicle_classes = vehicle_classes
        
        # Khởi tạo GMC chia sẻ dùng chung cho 3 tracker
        self.shared_gmc = GMC(method=args_global.cmc_method)
        
        # Tracker Person
        args_p = copy.deepcopy(args_global)
        args_p.with_reid = True
        args_p.fast_reid_config = getattr(args_global, 'person_reid_config', None)
        args_p.fast_reid_weights = getattr(args_global, 'person_reid_weights', None)
        self.tracker_person = BoTSORT(args_p, frame_rate=3)
        
        # Tracker Vehicle
        args_v = copy.deepcopy(args_global)
        args_v.with_reid = True
        args_v.fast_reid_config = getattr(args_global, 'vehicle_reid_config', None)
        args_v.fast_reid_weights = getattr(args_global, 'vehicle_reid_weights', None)
        self.tracker_vehicle = BoTSORT(args_v, frame_rate=3)
        
        # Tracker Other
        args_o = copy.deepcopy(args_global)
        args_o.with_reid = False
        self.tracker_other = BoTSORT(args_o, frame_rate=3)
        
    def update(self, dets_array, img):
        import numpy as np
        
        if len(dets_array) == 0:
            shared_warp = self.shared_gmc.apply(img, [])
            tracks_p = self.tracker_person.update(np.empty((0, 6), dtype=np.float32), img, shared_warp=shared_warp)
            tracks_v = self.tracker_vehicle.update(np.empty((0, 6), dtype=np.float32), img, shared_warp=shared_warp)
            tracks_o = self.tracker_other.update(np.empty((0, 6), dtype=np.float32), img, shared_warp=shared_warp)
            return tracks_p + tracks_v + tracks_o
            
        # Tính toán ma trận GMC một lần duy nhất với TẤT CẢ bounding boxes để che nhiễu đúng cách
        shared_warp = self.shared_gmc.apply(img, dets_array[:, :4])
            
        classes = dets_array[:, 5]
        
        mask_person = np.isin(classes, self.person_classes)
        mask_vehicle = np.isin(classes, self.vehicle_classes)
        mask_other = ~(mask_person | mask_vehicle)
        
        # Ghi nhớ lại chỉ số gốc (Global Index)
        indices_person = np.where(mask_person)[0]
        indices_vehicle = np.where(mask_vehicle)[0]
        indices_other = np.where(mask_other)[0]
        
        dets_person = dets_array[mask_person] if np.any(mask_person) else np.empty((0, 6), dtype=np.float32)
        dets_vehicle = dets_array[mask_vehicle] if np.any(mask_vehicle) else np.empty((0, 6), dtype=np.float32)
        dets_other = dets_array[mask_other] if np.any(mask_other) else np.empty((0, 6), dtype=np.float32)
        
        # 1. Update Person
        tracks_p = self.tracker_person.update(dets_person, img, shared_warp=shared_warp)
        for t in tracks_p:
            if getattr(t, "det_idx", -1) != -1:
                t.det_idx = int(indices_person[t.det_idx])
                
        # 2. Update Vehicle
        tracks_v = self.tracker_vehicle.update(dets_vehicle, img, shared_warp=shared_warp)
        for t in tracks_v:
            if getattr(t, "det_idx", -1) != -1:
                t.det_idx = int(indices_vehicle[t.det_idx])
                
        # 3. Update Other
        tracks_o = self.tracker_other.update(dets_other, img, shared_warp=shared_warp)
        for t in tracks_o:
            if getattr(t, "det_idx", -1) != -1:
                t.det_idx = int(indices_other[t.det_idx])
        
        return tracks_p + tracks_v + tracks_o

def run_tracking_pipeline(                        # Thêm detector ở đây
    img_root,
    output_file_path,
    yolo_model_path="yolo11n.pt",
    start_idx=None,
    end_idx=None,
    overwrite=False,
    save_only_last_frame=True,
):
    print("Đang load YOLO11 model từ:", yolo_model_path)
    from ultralytics import YOLO
    model = YOLO(yolo_model_path)

    print("Đang đọc danh sách thư mục từ:", img_root)
    all_folders = [f for f in os.listdir(img_root) if os.path.isdir(os.path.join(img_root, f))]
    all_folders = sorted(all_folders)

    if start_idx is not None or end_idx is not None:
        start = start_idx if start_idx is not None else 0
        end = end_idx if end_idx is not None else len(all_folders)
        all_folders = all_folders[start:end]
        print(f"Chạy test cho khoảng index [{start}:{end}]. Tổng số: {len(all_folders)} sequences.")
    else:
        print(f"Tổng số sequences: {len(all_folders)}")

    if overwrite and os.path.exists(output_file_path):
        os.remove(output_file_path)

    args = BoTSORTArgs()

    mode = "w" if overwrite else "a"
    with open(output_file_path, mode, encoding="utf-8") as out_f:

        for seq_idx, seq in enumerate(all_folders, 1):
            seq_path = os.path.join(img_root, seq)
            if not os.path.isdir(seq_path):
                continue
            frames = sorted(
                [f for f in os.listdir(seq_path) if f.endswith(".jpg")],
                key=lambda x: int(x.split(".")[0])
            )

            if len(frames) != 9:
                continue

            BaseTrack.clear_count()
            
            label_to_id = {"person": 0}
            id_to_label = {0: "person"}
            
            vehicle_labels = ['car_(automobile)', 'bus_(vehicle)', 'truck', 'minivan', 'pickup_truck', 'garbage_truck', 'cab_(taxi)', 'camper_(vehicle)']
            other_labels = []
            
            current_cls_id = 1
            vehicle_ids = []
            for v_label in vehicle_labels:
                label_to_id[v_label] = current_cls_id
                id_to_label[current_cls_id] = v_label
                vehicle_ids.append(current_cls_id)
                current_cls_id += 1
                
            for o_label in other_labels:
                label_to_id[o_label] = current_cls_id
                id_to_label[current_cls_id] = o_label
                current_cls_id += 1
                
            args.person_class_ids = [0]
            args.vehicle_class_ids = vehicle_ids
            
            tracker = DualReIDTrackerWrapper(args, args.person_class_ids, args.vehicle_class_ids)

            sequence_results = []
            previous_track_ids = set()
            
            for k in range(9):
                img_path = os.path.join(seq_path, frames[k])
                img = cv2.imread(img_path)
                if img is None:
                    continue
                
                img_h, img_w = img.shape[:2]
                focal = compute_focal_length(img_w, fov_deg=90)

                yolo_results = model(img, verbose=False)
                raw_frame_dets = []
                for box in yolo_results[0].boxes:
                    cls = int(box.cls[0].item())
                    label = model.names[cls]
                    conf = float(box.conf[0].item())
                    xyxyn = box.xyxyn[0].cpu().numpy().tolist()
                    
                    det = {
                        "folder_id": seq,
                        "frame_id": k,
                        "label": label,
                        "probs": conf,
                        "boxs": xyxyn
                    }
                    raw_frame_dets.append(det)

                valid_dets = []

                for idx, det in enumerate(raw_frame_dets):
                    x1_n, y1_n, x2_n, y2_n = det["boxs"]
                    box_px = [
                        x1_n * img_w,
                        y1_n * img_h,
                        x2_n * img_w,
                        y2_n * img_h,
                    ]
                    label = det["label"]
                    if label not in KEEP_LABELS:
                        continue
                    if label not in label_to_id:
                        label_to_id[label] = current_cls_id
                        id_to_label[current_cls_id] = label
                        current_cls_id += 1
                    cls_id = label_to_id[label]
                    valid_dets.append((idx, det, box_px, cls_id))

                if valid_dets:
                    dets_array = np.array(
                        [[d[2][0], d[2][1], d[2][2], d[2][3], d[1].get("probs", 0.9), d[3]] for d in valid_dets],
                        dtype=np.float32,
                    )
                else:
                    dets_array = np.empty((0, 6), dtype=np.float32)

                online_targets = tracker.update(dets_array, img)

                current_track_ids = set()
                for t in online_targets:
                    current_track_ids.add(t.track_id)
                    is_new = t.track_id not in previous_track_ids
                    
                    det_idx = getattr(t, "det_idx", -1)
                    if not (0 <= det_idx < len(valid_dets)):
                        continue
                        
                    orig_idx, orig_det, _, _ = valid_dets[det_idx]
                    box_n = orig_det["boxs"]
                    box_area_norm = (box_n[2] - box_n[0]) * (box_n[3] - box_n[1])
                    calc_tlbr = [
                        box_n[0] * img_w,
                        box_n[1] * img_h,
                        box_n[2] * img_w,
                        box_n[3] * img_h,
                    ]

                    motion = get_motion_info_v2(t, img_w, img_h)
                    azimuth_deg = compute_azimuth_from_bbox(calc_tlbr, img_w, focal)
                    y_bottom_normalized = box_n[3]
                    distance_norm = 1.0 - y_bottom_normalized
                    clock = azimuth_to_clock_10_to_2(azimuth_deg)
                    
                    if save_only_last_frame and k != 8:
                        continue
                        
                    sequence_results.append({
                        "folder_id": seq,
                        "frame_id": k,
                        "track_id": t.track_id,
                        "label": id_to_label.get(t.cls, "unknown"),
                        "boxs": box_n,
                        "center": [round((box_n[0]+box_n[2])/2, 4), round((box_n[1]+box_n[3])/2, 4)],
                        "area_norm": round(box_area_norm, 4),
                        "distance_norm": round(distance_norm, 2), 
                        "relative_position": clock,
                        "movement_angle": motion["angle"],
                        "speed_percent": motion["speed_percent"],
                    })

                previous_track_ids = current_track_ids

            if sequence_results:
                out_f.writelines(json.dumps(item, ensure_ascii=False) + "\n" for item in sequence_results)
                out_f.flush()
                
                if seq_idx % 50 == 0:
                    print(f"[{seq_idx}/{len(all_folders)}] Saved")


# Chạy Full dataset 

In [12]:
# ===== CELL: Full Dataset & Top 6 Filter =====
import json
import os
import pandas as pd

img_root = "/kaggle/input/datasets/cinminhcit/walkingawareness-wad-all-size/resized_images_only"
output_file = "/kaggle/working/results_botsort_26_07_ReID_107_labels.jsonl"
yolo_model_path = "yolo11n.pt" 

# Đếm tổng số sequence
print("Đang đọc danh sách thư mục từ:", img_root)
all_folders = [f for f in os.listdir(img_root) if os.path.isdir(os.path.join(img_root, f))]
# Chỉ test với 20 mẫu đầu tiên
all_folders = all_folders[:20]
TOTAL = len(all_folders)

BATCH_SIZE = 2000
print(f"Total sequences: {TOTAL}")

for start in range(0, TOTAL, BATCH_SIZE):

    end = min(start + BATCH_SIZE, TOTAL)

    print(f"\n========== Batch [{start}:{end}] ==========")

    run_tracking_pipeline(
        img_root=img_root,
        output_file_path=output_file,
        yolo_model_path=yolo_model_path,
        start_idx=start,
        end_idx=end,
        overwrite=(start == 0),
        save_only_last_frame=True,
    )

print("\nTracking Done!")
print("Saved raw output to:", output_file)

# ----------------------------------------------------
# 3. Lọc dữ liệu theo cơ chế tính điểm ở frame 8
# ----------------------------------------------------
print("\nBắt đầu lọc Top 6 objects cho mỗi sequence...")
df = pd.read_json(output_file, lines=True)

df_f8 = df[df["frame_id"] == 8].copy()

# Tính rank theo từng folder_id
# area_norm: ưu tiên kích thước lớn
df_f8["area_rank"] = (
    df_f8.groupby("folder_id", sort=False)["area_norm"]
         .rank(ascending=True)
)

# distance_norm: ưu tiên khoảng cách nhỏ
df_f8["distance_rank"] = (
    df_f8.groupby("folder_id", sort=False)["distance_norm"]
         .rank(ascending=False)
)

# Tính điểm (cân bằng giữa kích thước và khoảng cách)
df_f8["score"] = (
    0.5 * df_f8["area_rank"] +
    0.5 * df_f8["distance_rank"]
)

# Giữ nguyên thứ tự folder_id, chỉ sắp xếp object bên trong từng folder
top6_df = (
    df_f8
    .groupby("folder_id", sort=False, group_keys=False)
    .apply(lambda x: x.sort_values("score", ascending=False).head(6))
    .reset_index(drop=True)
)

# Loại bỏ các cột phụ trợ rank/score trước khi lưu
top6_df = top6_df.drop(columns=["area_rank", "distance_rank", "score"])

# Lưu ra file JSON mới
top6_output_file = output_file.replace(".jsonl", "_top6.jsonl")
top6_df.to_json(top6_output_file, orient="records", lines=True, force_ascii=False)
print(f"Đã lưu top 6 objects vào: {top6_output_file}")

print("\nDemo 10 mẫu json đầu tiên:")
with open(top6_output_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 10: break
        print(json.dumps(json.loads(line), indent=2, ensure_ascii=False))


Đang đọc danh sách thư mục từ: /kaggle/input/datasets/cinminhcit/walkingawareness-wad-all-size/resized_images_only
Total sequences: 20

========== Batch [0:20] ==========
Đang load YOLO11 model từ: yolo11n.pt
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Đang đọc danh sách thư mục từ: /kaggle/input/datasets/cinminhcit/walkingawareness-wad-all-size/resized_images_only
Chạy test cho khoảng index [0:20]. Tổng số: 20 sequences.


Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.
Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.
Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.
Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.
Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.
Skip loading parameter 'heads.weight' to the model due 


Tracking Done!
Saved raw output to: /kaggle/working/results_botsort_26_07_ReID_107_labels.jsonl

Bắt đầu lọc Top 6 objects cho mỗi sequence...
Đã lưu top 6 objects vào: /kaggle/working/results_botsort_26_07_ReID_107_labels_top6.jsonl

Demo 10 mẫu json đầu tiên:
{
  "folder_id": "20240914_13ab67a594ecb2636d92ef543c9231e2_1m28s.frame",
  "frame_id": 8,
  "track_id": 1,
  "label": "refrigerator",
  "boxs": [
    0.4302955866,
    0.336162895,
    0.9658107162,
    0.6823394895
  ],
  "center": [
    0.6981,
    0.5093
  ],
  "area_norm": 0.1854,
  "distance_norm": 0.32,
  "relative_position": "1 o'clock",
  "movement_angle": 9.56,
  "speed_percent": 0.32
}
{
  "folder_id": "20240914_13ab67a594ecb2636d92ef543c9231e2_1m34s.frame",
  "frame_id": 8,
  "track_id": 6,
  "label": "person",
  "boxs": [
    0.1047890484,
    0.4680617452,
    0.15000467,
    0.5233568549
  ],
  "center": [
    0.1274,
    0.4957
  ],
  "area_norm": 0.0025,
  "distance_norm": 0.48,
  "relative_position": "10 o'clo

/tmp/ipykernel_58/3338683458.py:70: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sort_values("score", ascending=False).head(6))
